In [1]:
# ============================================================
# 02_preprocessing.ipynb | CELL 1 — Load & Clean Data
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ── Load Engineered Dataset ───────────────────────────────────
DATA_PATH = "/teamspace/studios/this_studio/data/drug_reviews/medicine_engineered.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Loaded shape : {df.shape}")
print(f"Columns      : {list(df.columns)}\n")

# ── Step 1: Handle Missing Values ────────────────────────────
# Chemical Class & Action Class — ~44% missing → fill Unknown
df["Chemical Class"] = df["Chemical Class"].fillna("Unknown")
df["Action Class"]   = df["Action Class"].fillna("Unknown")

# Therapeutic Class — 0.03% missing → fill with mode
tc_mode = df["Therapeutic Class"].mode()[0]
df["Therapeutic Class"] = df["Therapeutic Class"].fillna(tc_mode)

print("Missing values after filling:")
print(df.isnull().sum().to_string())

# ── Step 2: Clean Text Columns ────────────────────────────────
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()
    text = text.replace("  ", " ")
    return text

df["all_side_effects"] = df["all_side_effects"].apply(clean_text)
df["all_uses"]         = df["all_uses"].apply(clean_text)
df["name"]             = df["name"].apply(clean_text)

# ── Step 3: Encode Habit Forming → Binary ─────────────────────
df["habit_forming"] = df["Habit Forming"].str.strip().str.lower().map(
    {"yes": 1, "no": 0}
).fillna(0).astype(int)

# ── Step 4: Encode Severity → Numeric Label ───────────────────
severity_map = {"MILD": 0, "MODERATE": 1, "CRITICAL": 2}
df["label"]   = df["severity"].map(severity_map)

# ── Step 5: Drop Original Redundant Columns ───────────────────
df.drop(columns=["Habit Forming", "severity"], inplace=True)

# ── Step 6: Final Clean DataFrame ────────────────────────────
print(f"\nFinal shape  : {df.shape}")
print(f"\nLabel distribution:")
label_map_inv = {0: "MILD", 1: "MODERATE", 2: "CRITICAL"}
dist = df["label"].value_counts().sort_index()
for k, v in dist.items():
    print(f"  {label_map_inv[k]:10s} ({k}) : {v:,}  ({v/len(df)*100:.1f}%)")

print(f"\nSample cleaned rows:")
print(df[["name", "all_side_effects", "se_count",
          "Therapeutic Class", "habit_forming", "label"]].head(3).to_string())

# ── Step 7: Save Cleaned Dataset ──────────────────────────────
SAVE_PATH = "/teamspace/studios/this_studio/data/drug_reviews/medicine_clean.csv"
df.to_csv(SAVE_PATH, index=False)
print(f"\nSaved cleaned dataset → {SAVE_PATH}")

Loaded shape : (248218, 9)
Columns      : ['name', 'all_side_effects', 'all_uses', 'se_count', 'Chemical Class', 'Habit Forming', 'Therapeutic Class', 'Action Class', 'severity']

Missing values after filling:
name                 0
all_side_effects     0
all_uses             0
se_count             0
Chemical Class       0
Habit Forming        0
Therapeutic Class    0
Action Class         0
severity             0

Final shape  : (248218, 9)

Label distribution:
  MILD       (0) : 126,918  (51.1%)
  MODERATE   (1) : 54,887  (22.1%)
  CRITICAL   (2) : 66,413  (26.8%)

Sample cleaned rows:
                       name                                                                                                                                                         all_side_effects  se_count Therapeutic Class  habit_forming  label
0  augmentin 625 duo tablet                                                                                                                                    

In [2]:
# ============================================================
# 02_preprocessing.ipynb | CELL 2 — Encode & Train/Test Split
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

# ── Load Clean Dataset ────────────────────────────────────────
DATA_PATH = "/teamspace/studios/this_studio/data/drug_reviews/medicine_clean.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded shape : {df.shape}\n")

# ── Step 1: Label Encode Categorical Columns ──────────────────
CAT_COLS   = ["Therapeutic Class", "Chemical Class", "Action Class"]
encoders   = {}

for col in CAT_COLS:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))
    encoders[col]    = le
    print(f"  {col:20s} → {df[col].nunique():>4} unique classes encoded")

# ── Step 2: Save Encoders ─────────────────────────────────────
ENCODER_DIR = "/teamspace/studios/this_studio/data/drug_reviews/"
os.makedirs(ENCODER_DIR, exist_ok=True)

with open(os.path.join(ENCODER_DIR, "label_encoders.pkl"), "wb") as f:
    pickle.dump(encoders, f)
print("\nEncoders saved → label_encoders.pkl")

# ── Step 3: Define Feature Groups ────────────────────────────
# Text features (for TF-IDF in feature engineering notebook)
TEXT_COLS  = ["name", "all_side_effects", "all_uses"]

# Structured features
STRUCT_COLS = [
    "se_count",
    "habit_forming",
    "Therapeutic Class_enc",
    "Chemical Class_enc",
    "Action Class_enc"
]

TARGET = "label"

print(f"\nText features    : {TEXT_COLS}")
print(f"Struct features  : {STRUCT_COLS}")
print(f"Target           : {TARGET}")

# ── Step 4: Stratified Train / Test Split (80:20) ─────────────
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df[TARGET]
)

print(f"\nTrain shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")

# ── Step 5: Verify Class Distribution ────────────────────────
label_map = {0: "MILD", 1: "MODERATE", 2: "CRITICAL"}

print("\nClass distribution in TRAIN:")
for k, v in train_df[TARGET].value_counts().sort_index().items():
    print(f"  {label_map[k]:10s}: {v:,}  ({v/len(train_df)*100:.1f}%)")

print("\nClass distribution in TEST:")
for k, v in test_df[TARGET].value_counts().sort_index().items():
    print(f"  {label_map[k]:10s}: {v:,}  ({v/len(test_df)*100:.1f}%)")

# ── Step 6: Save Train / Test Sets ────────────────────────────
train_df.to_csv(os.path.join(ENCODER_DIR, "train.csv"), index=False)
test_df.to_csv(os.path.join(ENCODER_DIR,  "test.csv"),  index=False)

print(f"\nSaved → train.csv  ({len(train_df):,} rows)")
print(f"Saved → test.csv   ({len(test_df):,} rows)")

Loaded shape : (248218, 9)

  Therapeutic Class    →   22 unique classes encoded
  Chemical Class       →  873 unique classes encoded
  Action Class         →  432 unique classes encoded

Encoders saved → label_encoders.pkl

Text features    : ['name', 'all_side_effects', 'all_uses']
Struct features  : ['se_count', 'habit_forming', 'Therapeutic Class_enc', 'Chemical Class_enc', 'Action Class_enc']
Target           : label

Train shape : (198574, 12)
Test  shape : (49644, 12)

Class distribution in TRAIN:
  MILD      : 101,534  (51.1%)
  MODERATE  : 43,910  (22.1%)
  CRITICAL  : 53,130  (26.8%)

Class distribution in TEST:
  MILD      : 25,384  (51.1%)
  MODERATE  : 10,977  (22.1%)
  CRITICAL  : 13,283  (26.8%)

Saved → train.csv  (198,574 rows)
Saved → test.csv   (49,644 rows)
